# Data Cleaning and Fusion

This notebook standardizes stroke datasets from multiple sources and merges them into a unified dataset with consistent schema and data types.

## 1. Import Required Libraries

Load pandas and numpy for data manipulation and analysis.

In [4]:
import pandas as pd
import numpy as np

## 2. Load Raw Data

Load the three stroke datasets from CSV files: Kaggle, Full-filled, and Synthetic sources.

In [7]:
df_kaggle = pd.read_csv('../data/raw/stroke_kaggle.csv')
df_fullfilled = pd.read_csv('../data/raw/stroke_fullfilled.csv')
df_synthetic = pd.read_csv('../data/raw/stroke_synthetic.csv')

## 3. Define Target Schema

Define the standard column names and data types for the unified dataset.

In [8]:
TARGET_COLS = {
    'gender': 'object',
    'age': 'float64',
    'hypertension': 'int64',
    'heart_disease': 'int64',
    'ever_married': 'object',
    'work_type': 'object',
    'residence_type': 'object',
    'avg_glucose_level': 'float64',
    'bmi': 'float64',
    'smoking_status': 'object',
    'stroke': 'int64'
}

## 4. Standardize Dataset Function

Create a function to standardize schema, data types, and values across all datasets.

In [9]:
def standardize_dataset(df, source_name):
    """
    Hàm chuẩn hóa schema và data types cho một dataframe.
    """
    df_clean = df.copy()
    
    # --- A. XỬ LÝ TÊN CỘT (SCHEMA) ---
    # 1. Xóa cột ID nếu có (vì không dùng để train)
    if 'id' in df_clean.columns:
        df_clean = df_clean.drop(columns=['id'])
    
    # 2. Đưa tên cột về chữ thường (lowercase)
    df_clean.columns = df_clean.columns.str.lower()
    
    # 3. Mapping tên cột (nếu dataset có tên khác biệt)
    # Ví dụ: nếu có dataset dùng 'res_type' thay vì 'residence_type'
    rename_map = {
        'res_type': 'residence_type',
        'sex': 'gender',
        'glucose': 'avg_glucose_level'
    }
    df_clean = df_clean.rename(columns=rename_map)
    
    # --- B. XỬ LÝ KIỂU DỮ LIỆU (DATA TYPES) ---
    for col, dtype in TARGET_COLS.items():
        if col in df_clean.columns:
            # Xử lý đặc biệt cho BMI (vì bộ gốc Kaggle có 'N/A' là string)
            if col == 'bmi' and df_clean[col].dtype == 'object':
                # Chuyển 'N/A' thành NaN thực sự của numpy để máy hiểu là missing
                df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
            
            # Ép kiểu dữ liệu chuẩn
            try:
                df_clean[col] = df_clean[col].astype(dtype)
            except ValueError:
                print(f"⚠️ Cảnh báo: Không thể ép kiểu cột '{col}' trong bộ {source_name}")

    # --- C. CHUẨN HÓA GIÁ TRỊ (VALUES) ---
    # Đảm bảo gender chỉ là Male/Female/Other (viết hoa chữ đầu)
    if 'gender' in df_clean.columns:
        df_clean['gender'] = df_clean['gender'].str.capitalize()
        
    # Đảm bảo residence_type chuẩn (Urban/Rural)
    if 'residence_type' in df_clean.columns:
        df_clean['residence_type'] = df_clean['residence_type'].str.title() # Viết hoa chữ cái đầu

    # --- D. GẮN NHÃN NGUỒN ---
    df_clean['source'] = source_name
    
    # Chỉ giữ lại các cột nằm trong TARGET_COLS + source
    final_cols = [c for c in df_clean.columns if c in TARGET_COLS or c == 'source']
    return df_clean[final_cols]

## 5. Apply Standardization

Apply the standardization function to all three datasets.

In [10]:
print("⏳ Đang chuẩn hóa dữ liệu...")
df_kaggle_clean = standardize_dataset(df_kaggle, 'original')
df_full_clean = standardize_dataset(df_fullfilled, 'fullfilled')
df_synth_clean = standardize_dataset(df_synthetic, 'synthetic')

print(f"✅ Kaggle shape: {df_kaggle_clean.shape}")
print(f"✅ Full-filled shape: {df_full_clean.shape}")
print(f"✅ Synthetic shape: {df_synth_clean.shape}")

⏳ Đang chuẩn hóa dữ liệu...
✅ Kaggle shape: (5110, 12)
✅ Full-filled shape: (4981, 12)
✅ Synthetic shape: (50000, 12)


## 6. Merge All Datasets

Combine the three standardized datasets into a single unified dataset.

In [11]:
stroke_all = pd.concat([df_kaggle_clean, df_full_clean, df_synth_clean], axis=0, ignore_index=True)

## 7. Final Verification

Check the merged dataset's information and data types.

In [12]:
print("\n--- INFO SAU KHI MERGE ---")
print(stroke_all.info())


--- INFO SAU KHI MERGE ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60091 entries, 0 to 60090
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             60091 non-null  object 
 1   age                60091 non-null  float64
 2   hypertension       60091 non-null  int64  
 3   heart_disease      60091 non-null  int64  
 4   ever_married       60091 non-null  object 
 5   work_type          60091 non-null  object 
 6   residence_type     60091 non-null  object 
 7   avg_glucose_level  60091 non-null  float64
 8   bmi                57390 non-null  float64
 9   smoking_status     60091 non-null  object 
 10  stroke             60091 non-null  int64  
 11  source             60091 non-null  object 
dtypes: float64(3), int64(3), object(6)
memory usage: 5.5+ MB
None


## 7. Save processed data

Save processed data for using in next step

In [13]:
stroke_all.to_csv('../data/processed/stroke_all.csv', index=False)